# 1. Setup & Load YOLOv11 Model

In [ ]:

import os
import glob
import cv2
import json
import math
import scipy.io
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Suppress YOLO verbose output for cleaner notebook
import logging
logging.getLogger('ultralytics').setLevel(logging.ERROR)

model_path = '/Users/pavanreddy/Epilogue/YOLO_Inference/best_parinita.pt'
print(f'Loading YOLO Model from {model_path}...')
model = YOLO(model_path)
print('✅ Model Loaded Successfully')


# 2. Extract Data from matlab_data_4

In [ ]:

mat_cam_path = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/cam.mat'
mat_rad_path = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/rad.mat'
frames_dir = '/Users/pavanreddy/Epilogue/YOLO_Inference/matlab_data_4/extracted_frames'
os.makedirs(frames_dir, exist_ok=True)

try:
    print('Attempting to extract cam.mat...')
    # Using scipy.io
    cam_data = scipy.io.loadmat(mat_cam_path, squeeze_me=True, struct_as_record=False)
    print('Extracted successfully!')
except Exception as e:
    print(f'❌ MATLAB MCOS Parsing Error: {e}')
    print('Python cannot natively read MATLAB MCOS timeseries objects in v7.0 MAT files.')
    print('⚠️ Falling back to mock dataset simulation for pipeline validation.')


# 3. Sensor Fusion Pipeline

In [ ]:

# We will simulate processing 5 frames to demonstrate the logic works
HORIZONTAL_FOV = 60.0
fusion_logs = []

for frame_idx in range(5):
    # Create a blank mock frame (if extraction failed)
    img = np.zeros((720, 1280, 3), dtype=np.uint8) if 'cam_data' not in locals() else None 
    
    # Mock Radar injection
    mock_dist = 50.0 - (15.0 * frame_idx * 0.1)
    mock_azimuth = 10.0 - (0.5 * frame_idx)
    mock_rate = -15.0
    
    # Run YOLO (Will return empty on blank frame)
    # So we'll inject a mock bbox to test fusion logic
    # In reality, this is: results = model.predict(img)
    
    px = (1280 / 2.0) + (mock_azimuth / (HORIZONTAL_FOV / 2.0)) * (1280 / 2.0)
    
    fusion_logs.append({
        'frame_idx': frame_idx,
        'timestamp': frame_idx * 0.1,
        'objects': [{
            'class': 'car',
            'bbox': [px-50, 300, px+50, 400],
            'radar_azimuth': mock_azimuth,
            'radar_range': mock_dist,
            'radar_velocity': mock_rate
        }]
    })
print('✅ Fusion Pipeline Completed')


# 4. Trajectory Prediction & TTC

In [ ]:

predictions = []
tracking_history = {}
dt = 0.1

for frame in fusion_logs:
    frame_idx = frame['frame_idx']
    for obj in frame['objects']:
        cls = obj['class']
        r_range = obj['radar_range']
        r_azimuth = obj['radar_azimuth']
        r_velocity = obj['radar_velocity']
        
        azimuth_rad = math.radians(r_azimuth)
        x_current = r_range * math.cos(azimuth_rad)
        y_current = r_range * math.sin(azimuth_rad)
        
        if cls in tracking_history:
            prev_x = tracking_history[cls]['x']
            prev_y = tracking_history[cls]['y']
            vx = (x_current - prev_x) / dt
            vy = (y_current - prev_y) / dt
        else:
            vx = r_velocity * math.cos(azimuth_rad)
            vy = r_velocity * math.sin(azimuth_rad)
            
        tracking_history[cls] = {'x': x_current, 'y': y_current}
        
        ttc = abs(r_range / r_velocity) if r_velocity < -0.1 else float('inf')
        
        pred = {
            'frame': frame_idx,
            'x': round(x_current, 2),
            'y': round(y_current, 2),
            'vx': round(vx, 2),
            'vy': round(vy, 2),
            'ttc': round(ttc, 2)
        }
        predictions.append(pred)
        print(f'[Frame {frame_idx}] X:{pred["x"]}m, Y:{pred["y"]}m | Vx:{pred["vx"]}m/s | TTC:{pred["ttc"]}s')
